# AI Engineering — First Principles Reference

**AIcademy Knowledgebase | AI Engineer Track**

This notebook is the reference for the AI Engineer goal track. It assumes you've completed Stages 0–6 of the Python curriculum.

**Sections:**
- [AE-1: What is an LLM?](#ae-1)
- [AE-2: Calling AI APIs](#ae-2)
- [AE-3: Prompt Engineering](#ae-3)
- [AE-4: Embeddings and Vector Search](#ae-4)
- [AE-5: Retrieval-Augmented Generation (RAG)](#ae-5)
- [AE-6: AI Agents](#ae-6)

---
## AE-1: What is an LLM? <a id='ae-1'></a>

### The mental model

A **Large Language Model (LLM)** is a program trained on a massive amount of text (books, websites, code, conversations) that learned to predict: *"given this text, what word comes next?"*

That sounds simple. But when you train this prediction task on enough data with enough compute, something surprising happens: the model develops a rich internal representation of language, facts, reasoning, and even code. ChatGPT, Claude, and Gemini are all LLMs.

**Key concepts:**
- **Token** — the unit an LLM thinks in. Not exactly a word — roughly 3–4 characters on average. "Hello world" is 2 tokens.
- **Context window** — how much text the model can "see" at once. GPT-4 has ~128,000 tokens of context.
- **Temperature** — controls randomness. 0 = deterministic (same input → same output). 1+ = creative/random.
- **Parameters** — the "weights" a model learned during training. GPT-4 is estimated at ~1 trillion parameters.

### What LLMs are NOT
- They are not searching the internet in real time (unless you give them a tool for that)
- They are not "thinking" the way humans do
- They can be confidently wrong (called "hallucination")
- Their knowledge has a cutoff date

---
## AE-2: Calling AI APIs <a id='ae-2'></a>

### What is an API?

An **API (Application Programming Interface)** is a door into someone else's software. When you call the OpenAI API, you're sending a message to OpenAI's servers saying "run this text through your model and send me the result."

You don't need to understand how the model works internally — you just need to know how to format your request and read the response.

### The anatomy of an API call

In [ ]:
# First: pip install openai
# Make sure OPENAI_API_KEY is set in your .env file

import os
from openai import OpenAI

# Load your API key from the environment (never hardcode it)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Make a request
response = client.chat.completions.create(
    model="gpt-4o-mini",       # which model to use
    messages=[
        {
            "role": "system",   # sets the AI's behavior/persona
            "content": "You are a helpful coding tutor."
        },
        {
            "role": "user",     # the human's message
            "content": "What is a Python list?"
        }
    ],
    temperature=0.7,           # 0=deterministic, 1=more creative
    max_tokens=500             # limit response length
)

# Extract the text from the response
answer = response.choices[0].message.content
print(answer)

### Message roles explained

| Role | Purpose |
|---|---|
| `system` | Sets the AI's behavior, persona, and constraints for the whole conversation |
| `user` | A message from the human |
| `assistant` | A previous response from the AI (used to maintain conversation history) |

### Building a conversation (multi-turn)

In [ ]:
# Maintain conversation history by keeping track of all messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def chat(user_input):
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    ai_reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": ai_reply})
    return ai_reply

# Now the AI remembers context across turns
print(chat("My name is Alex."))
print(chat("What's my name?"))  # Will correctly say "Alex"

---
## AE-3: Prompt Engineering <a id='ae-3'></a>

### What is prompt engineering?

Prompt engineering is the practice of crafting your input to an LLM to get better, more reliable outputs. It's not magic — it's communication design.

The core insight: **LLMs are very good at pattern completion**. Your prompt sets up a pattern; the model completes it. The more clearly you define the pattern, the better the completion.

### Key techniques

In [ ]:
# Technique 1: Be specific about format
vague_prompt = "Tell me about Python lists."

specific_prompt = """
Explain Python lists to a complete beginner.
Format your response as:
1. A one-sentence definition using a real-world analogy
2. A code example with 3 lines showing how to create, access, and add to a list
3. One common mistake beginners make
"""

# Specific prompts → specific, usable outputs

In [ ]:
# Technique 2: Role prompting (system message)
system_message = """
You are a senior software engineer conducting a technical interview.
Ask one coding question at a time. After the candidate answers,
give specific feedback on correctness, edge cases, and complexity.
Do not reveal the ideal solution until the candidate asks.
"""

In [ ]:
# Technique 3: Few-shot prompting — show examples
few_shot_prompt = """
Classify the sentiment of each review as Positive, Negative, or Neutral.

Review: "This product is amazing! Exceeded all expectations."
Sentiment: Positive

Review: "Arrived broken. Terrible quality."
Sentiment: Negative

Review: "It's okay. Does what it says."
Sentiment: Neutral

Review: "I've had better but can't complain for the price."
Sentiment:
"""
# The model continues the pattern → "Neutral"

In [ ]:
# Technique 4: Chain of thought — ask the model to reason step by step
cot_prompt = """
A store has 12 apples. They sell 3 in the morning and receive a delivery
of 8 more. Then they sell half of what they have. How many apples remain?

Think through this step by step before giving the final answer.
"""
# Adding "think step by step" dramatically improves accuracy on reasoning tasks

### Structured output (JSON)

For production AI systems, you often need the model to return structured data, not prose.

In [ ]:
import json

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": """
            Extract the key information from this job posting and return it as JSON:
            'We are looking for a Senior Python Engineer with 5+ years of experience
            in backend development. The role pays $150,000-$180,000 and is remote.'
            
            Return: {"title": ..., "level": ..., "years_exp": ..., "salary_min": ..., "salary_max": ..., "remote": ...}
            """
        }
    ],
    response_format={"type": "json_object"}  # force JSON output
)

data = json.loads(response.choices[0].message.content)
print(data["title"])  # "Senior Python Engineer"

---
## AE-4: Embeddings and Vector Search <a id='ae-4'></a>

### The mental model

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text. Text with similar meaning has vectors that are close together in space.

Examples:
- "dog" and "puppy" → vectors very close together
- "dog" and "car" → vectors far apart
- "I love Python" and "Python is my favorite language" → vectors close together

This lets you do **semantic search** — find content by meaning, not just keyword matching.

In [ ]:
# Generating an embedding
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Python is a great programming language for beginners."
)

embedding = response.data[0].embedding
print(f"Embedding length: {len(embedding)}")  # 1536 numbers
print(f"First few values: {embedding[:5]}")

In [ ]:
import numpy as np

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(vec1, vec2):
    """Measure how similar two vectors are. Returns 0 (different) to 1 (identical)."""
    v1, v2 = np.array(vec1), np.array(vec2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Semantic search example
documents = [
    "Python lists store ordered collections of items.",
    "How to bake chocolate chip cookies.",
    "Variables in Python point to values in memory.",
    "The best hiking trails in Patagonia."
]

query = "What are Python data structures?"
query_embedding = get_embedding(query)

similarities = []
for doc in documents:
    doc_embedding = get_embedding(doc)
    score = cosine_similarity(query_embedding, doc_embedding)
    similarities.append((score, doc))

similarities.sort(reverse=True)
print("Most relevant:", similarities[0][1])
# Will return the Python-related documents, not cookies or hiking

---
## AE-5: Retrieval-Augmented Generation (RAG) <a id='ae-5'></a>

### The problem RAG solves

LLMs have a knowledge cutoff — they don't know about recent events, your private documents, or your company's internal data. RAG solves this by giving the model relevant information at query time.

**How RAG works:**
1. **Index** your documents by converting them to embeddings and storing them
2. When a user asks a question, **embed the question** and find the most similar documents
3. **Inject** those documents into the prompt as context
4. The LLM generates an answer **grounded** in the retrieved documents

In [ ]:
# Minimal RAG implementation (no vector database — just for learning)

# Step 1: Your knowledge base (in production, this would be thousands of docs)
knowledge_base = [
    "AIcademy is an adaptive coding curriculum that teaches Python from first principles.",
    "The AIcademy learner profile is stored in LEARNER_PROFILE.md.",
    "AIcademy has 6 stages: Stage 0 (Mental Models) through Stage 6 (Leveling Up).",
    "The competency map tracks four states: not started, exposed, demonstrated, mastered."
]

# Step 2: Index (embed all documents)
indexed_docs = [(doc, get_embedding(doc)) for doc in knowledge_base]

# Step 3: Retrieve — find the most relevant document for a user query
def retrieve(query, top_k=2):
    query_emb = get_embedding(query)
    scored = [(cosine_similarity(query_emb, emb), doc) for doc, emb in indexed_docs]
    scored.sort(reverse=True)
    return [doc for _, doc in scored[:top_k]]

# Step 4: Generate — use retrieved docs as context
def rag_answer(user_question):
    relevant_docs = retrieve(user_question)
    context = "\n".join(relevant_docs)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"Answer the user's question using only the following context:\n\n{context}"
            },
            {"role": "user", "content": user_question}
        ]
    )
    return response.choices[0].message.content

print(rag_answer("How does AIcademy track my progress?"))

---
## AE-6: AI Agents <a id='ae-6'></a>

### What is an AI agent?

An **AI agent** is an LLM that can take actions — not just generate text, but call functions, search the web, run code, or interact with APIs — to complete a multi-step task.

The key ingredients:
- **LLM** — the reasoning engine
- **Tools** — functions the LLM can call
- **Loop** — the agent runs in a loop: think → act → observe → think again

### Tool calling (function calling)

In [ ]:
import json

# Define tools the AI can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Your actual function implementation
def get_weather(city):
    # In production, this would call a real weather API
    return {"city": city, "temperature": 72, "condition": "Sunny"}

# Ask the model something that requires using the tool
messages = [{"role": "user", "content": "What's the weather in New York?"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

# Check if the model wants to call a tool
if response.choices[0].finish_reason == "tool_calls":
    tool_call = response.choices[0].message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    # Call the actual function
    result = get_weather(**function_args)
    
    # Send the result back to the model
    messages.append(response.choices[0].message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result)
    })
    
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    print(final_response.choices[0].message.content)
    # "The weather in New York is currently 72°F and Sunny."